In [1]:
import openai
import pandas as pd
import os

In [4]:
prompts = os.listdir('Prompts')
prompts = [p for p in prompts if p.endswith('.xlsx')]
prompt_dict = {}
for prompt in prompts:
    prompt_dict[f'{prompt}'.replace('.xlsx','_df')] = pd.read_excel(f'Prompts/{prompt}')

## Testing out the APIs

In [ ]:
from google import genai

client = genai.Client(api_key='')

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="Explain how AI works in a few words",
)

print(response.text)

AI enables computers to **learn from data** to **identify patterns** and then **make decisions or predictions**.


In [ ]:
    from openai import OpenAI

    client = OpenAI(api_key='') # Automatically reads API key from OPENAI_API_KEY environment variable

    chat_completion = client.chat.completions.create(
        messages=[
            {
                "role": "user",
                "content": "Say this is a test",
            }
        ],
        model="gpt-3.5-turbo",
    )

    print(chat_completion.choices[0].message.content)


RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

## Prepare AI prompt code

In [ ]:
from google import genai

def gemini_generate_content(prompt):
    client = genai.Client(api_key='')
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt,
    )
    return response.text

In [6]:
chatbot_dict = {
    'gemini': gemini_generate_content,}

In [13]:
for key, value in prompt_dict.items():
    print(key)
    with open(f'Prompts/{key}.csv') as f:
        current_df = pd.read_csv(f)
        if 'gemini Response' in current_df.columns:
            print(f"Skipping {key} as it already has a gemini response.")
            continue
        else:
            for chatbot, function in chatbot_dict.items():
                value[f'{chatbot} Response'] = value['Example Prompt'].apply(function)
        # response = function(value['Example Prompt'])
        # print(f"Response: {response}\n")

Translation_df
Email_drafting_df
Fact_retrieval_df
Programming_df
Dialogue_simulation_df
Math_problem_solving_df
Summarization_df
Argument_analysis_df
Story_writing_df
Skipping Story_writing_df as it already has a gemini response.
Poetry_generation_df
Skipping Poetry_generation_df as it already has a gemini response.


In [17]:
prompt_dict['Argument_analysis_df']

,Prompting Technique,Example Prompt,gemini Response
0,Zero-Shot Prompting,Analyze the following argument and identify an...,"The argument ""We should ban all video games be..."
1,Few-Shot Prompting,"Example 1: “Argument: ‘If we allow A, then B w...","Argument: “Everyone is doing it, so it must be..."
2,Chain-of-Thought Prompting,Let’s break this down. What is the claim? What...,Excellent! This is a fantastic framework for c...
3,Iterative Refinement Prompting,Initial analysis: “This is a bad argument.”\nR...,This is an excellent illustration of moving fr...
4,Tree-of-Thought Prompting,Present multiple possible critiques of an argu...,"Let's consider a common, multi-faceted argumen..."
5,Role-Playing Prompting,You are a debate coach. Evaluate this argument...,"Alright, debater! I'm ready to hear it. Lay yo..."
6,Self-Critique Prompting,Write an analysis of this argument: “Freedom o...,"Here's an analysis of the argument, followed b..."
7,Retrieval-Augmented Prompting,Refer to a list of known logical fallacies to ...,"The argument, ""If we legalize X, society will ..."
8,Prompt Chaining,Prompt 1: Extract the claim.\nPrompt 2: Extrac...,Please provide the text you would like me to a...
9,Meta-Prompting,Create a prompt that helps a student break dow...,Here's a comprehensive prompt designed to help...


In [18]:
for key, value in prompt_dict.items():
    if f'{key}.csv' in os.listdir('Prompts'):
        os.remove(f'Prompts/{key}.csv')    
        value.to_csv(f'Prompts/{key}.csv', index=False, quotechar='"')
        print(f"Saved {key}.csv with responses.")

Saved Translation_df.csv with responses.
Saved Email_drafting_df.csv with responses.
Saved Fact_retrieval_df.csv with responses.
Saved Programming_df.csv with responses.
Saved Dialogue_simulation_df.csv with responses.
Saved Math_problem_solving_df.csv with responses.
Saved Summarization_df.csv with responses.
Saved Argument_analysis_df.csv with responses.
Saved Story_writing_df.csv with responses.
Saved Poetry_generation_df.csv with responses.


## Scoring Functions

In [5]:
import time

def test_passed(code, tests):
    """
    Executes the given code and runs each test expression.
    Returns True if all tests pass, otherwise False.
    """
    try:
        local_env = {}
        exec(code, local_env)
        for test in tests:
            if not eval(test, local_env):
                return False
        return True
    except Exception as e:
        print(f"Test failed with error: {e}")
        return False

def runtime_success(code):
    """
    Attempts to run the code and returns True if no exception is raised.
    """
    try:
        exec(code)
        return True
    except Exception as e:
        print(f"Runtime error: {e}")
        return False

def runtime_time(code):
    """
    Measures and returns the time in seconds it takes to execute the code.
    """
    start = time.time()
    try:
        exec(code)
    except Exception as e:
        print(f"Execution failed: {e}")
    end = time.time()
    return end - start

def correctness_success(code, expected_output):
    """
    Runs the code and compares the value of the 'result' variable to expected_output.
    Returns True if they match, otherwise False.
    """
    local_env = {}
    try:
        exec(code, local_env)
        return local_env.get("result") == expected_output
    except Exception as e:
        print(f"Correctness check failed: {e}")
        return False

In [9]:
import pandas as pd
programming_df = pd.read_csv('Prompts/Programming_df.csv')
programming_tests = [correctness_success, runtime_success, runtime_time, test_passed]
for test in programming_tests:
    programming_df[f'{test.__name__}'] = programming_df['gemini Response'].apply(lambda x: test(x, programming_df['gemini Response']) if test == correctness_success else test(x))
programming_df.to_csv('Prompts/Programming_df_full.csv', index=False, quotechar='"')

Correctness check failed: unterminated string literal (detected at line 3) (<string>, line 3)
Correctness check failed: invalid syntax (<string>, line 1)
Correctness check failed: unterminated string literal (detected at line 1) (<string>, line 1)
Correctness check failed: unterminated string literal (detected at line 1) (<string>, line 1)
Correctness check failed: unterminated string literal (detected at line 3) (<string>, line 3)
Correctness check failed: unterminated string literal (detected at line 79) (<string>, line 79)
Correctness check failed: unterminated string literal (detected at line 3) (<string>, line 3)
Correctness check failed: unterminated string literal (detected at line 95) (<string>, line 95)
Correctness check failed: unterminated string literal (detected at line 1) (<string>, line 1)
Correctness check failed: unterminated string literal (detected at line 1) (<string>, line 1)
Runtime error: unterminated string literal (detected at line 3) (<string>, line 3)
Runtime

TypeError: test_passed() missing 1 required positional argument: 'tests'

In [2]:
from rouge_score import rouge_scorer
from bert_score import score as bert_score_fn

def rouge1_f(summary, reference):
    """
    Computes ROUGE-1 F1 score between summary and reference.
    """
    scorer = rouge_scorer.RougeScorer(['rouge1'], use_stemmer=True)
    scores = scorer.score(reference, summary)
    return scores['rouge1'].fmeasure

def rougeL_f(summary, reference):
    """
    Computes ROUGE-L F1 score between summary and reference.
    """
    scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
    scores = scorer.score(reference, summary)
    return scores['rougeL'].fmeasure

def bertscore(summary, reference):
    """
    Computes BERTScore F1 between summary and reference.
    """
    P, R, F1 = bert_score_fn([summary], [reference], lang="en", verbose=False)
    return F1[0].item()

def summary_length(summary):
    """
    Returns number of words in the summary.
    """
    return len(summary.split())

def reference_length(reference):
    """
    Returns number of words in the reference.
    """
    return len(reference.split())

def length_ratio(summary, reference):
    """
    Returns ratio of summary length to reference length.
    """
    ref_len = reference_length(reference)
    return summary_length(summary) / ref_len if ref_len > 0 else 0

/usr/local/python/3.12.1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [1]:
from sacrebleu import corpus_bleu, corpus_chrf
from comet import download_model, load_from_checkpoint

# Load COMET model once globally
comet_model_path = download_model("Unbabel/wmt22-comet-da")
comet_model = load_from_checkpoint(comet_model_path)

def bleu_score(candidate, reference):
    """
    Computes BLEU score between a candidate and a reference translation.
    """
    return corpus_bleu([candidate], [[reference]]).score

def chrf_score(candidate, reference):
    """
    Computes chrF score (character n-gram F-score) between a candidate and a reference.
    """
    return corpus_chrf([candidate], [[reference]]).score

def comet_score(source, candidate, reference):
    """
    Computes COMET score, which uses a pretrained model to evaluate translation quality.
    """
    data = [{
        "src": source,
        "mt": candidate,
        "ref": reference
    }]
    result = comet_model.predict(data, batch_size=1, gpus=0)
    return result[0]["score"]

/usr/local/python/3.12.1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/usr/local/python/3.12.1/lib/python3.12/site-packages/torchmetrics/utilities/imports.py:23: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution
Fetching 5 files: 100%|██████████| 5/5 [00:00<00:00, 48321.47it/s]
Lightning automatically upgraded your loaded checkpoint from v1.8.3.post1 to v2.5.2. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../../../home/codespace/.cache/huggingface/hub/models--Unbabel--wmt22-comet-da

: 

In [1]:
import sympy

def symbolic_match(expr1, expr2):
    """
    Returns True if expr1 and expr2 are symbolically equivalent.
    Accepts expressions as strings (e.g., "x**2 + 2*x + 1", "(x+1)**2").
    """
    try:
        x = sympy.symbols("x")  # define default variable for parsing
        simplified_expr1 = sympy.simplify(expr1)
        simplified_expr2 = sympy.simplify(expr2)
        return simplified_expr1 == simplified_expr2
    except Exception as e:
        print(f"Symbolic match error: {e}")
        return False

def numeric_match(val1, val2, tol=1e-4):
    """
    Returns True if val1 and val2 are numerically close within tolerance `tol`.
    Accepts numbers or numeric strings.
    """
    try:
        return abs(float(val1) - float(val2)) < tol
    except Exception as e:
        print(f"Numeric match error: {e}")
        return False

In [2]:
import nltk
from nltk.tokenize import sent_tokenize
import textstat
import language_tool_python
from transformers import pipeline
import numpy as np

# Initialize tools
tool = language_tool_python.LanguageTool('en-US')
sentiment = pipeline("sentiment-analysis")

def grammar_errors(text):
    """
    Returns number of grammar and spelling issues found.
    """
    matches = tool.check(text)
    return len(matches)

def flesch_reading_ease(text):
    """
    Returns the Flesch Reading Ease score.
    Higher scores = easier to read.
    """
    return textstat.flesch_reading_ease(text)

def readability_grade(text):
    """
    Returns estimated U.S. school grade level needed to understand text.
    """
    return textstat.text_standard(text, float_output=True)

def tone(text):
    """
    Returns 'POSITIVE', 'NEGATIVE', or 'NEUTRAL' sentiment label.
    """
    result = sentiment(text)[0]['label']
    return result.upper()

def clarity_score(text):
    """
    Returns average sentence length in words (lower is generally clearer).
    """
    sentences = sent_tokenize(text)
    if not sentences:
        return 0
    lengths = [len(sentence.split()) for sentence in sentences]
    return np.mean(lengths)

/usr/local/python/3.12.1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Unzipping /tmp/tmpr0j_olzc.zip to /home/codespace/.cache/language_tool_python.
Downloaded https://internal1.languagetool.org/snapshots/LanguageTool-latest-snapshot.zip to /home/codespace/.cache/language_tool_python.
No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f (https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.
Device set to use cpu


In [13]:
import spacy
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import pipeline

# Load spaCy English model and transformer
nlp = spacy.load("en_core_web_sm")
embedding_pipeline = pipeline("feature-extraction")

# ---------- Entity Extraction Functions ----------
def extract_entities(text):
    doc = nlp(text)
    return set(ent.text.lower() for ent in doc.ents)

def entity_recall(predicted, reference):
    """
    Fraction of reference entities that appear in prediction.
    """
    pred_ents = extract_entities(predicted)
    ref_ents = extract_entities(reference)
    return len(pred_ents & ref_ents) / len(ref_ents) if ref_ents else 0.0

def entity_precision(predicted, reference):
    """
    Fraction of predicted entities that are in reference.
    """
    pred_ents = extract_entities(predicted)
    ref_ents = extract_entities(reference)
    return len(pred_ents & ref_ents) / len(pred_ents) if pred_ents else 0.0

def entity_overlap_count(predicted, reference):
    """
    Total number of overlapping named entities.
    """
    return len(extract_entities(predicted) & extract_entities(reference))


# ---------- Keyword Matching Functions ----------
def extract_keywords(text):
    vec = CountVectorizer(stop_words="english").fit([text])
    return set(vec.get_feature_names_out())

def keyword_recall(predicted, reference):
    """
    Fraction of reference keywords that appear in prediction.
    """
    pred_kw = extract_keywords(predicted)
    ref_kw = extract_keywords(reference)
    return len(pred_kw & ref_kw) / len(ref_kw) if ref_kw else 0.0

def keyword_precision(predicted, reference):
    """
    Fraction of predicted keywords that are in reference.
    """
    pred_kw = extract_keywords(predicted)
    ref_kw = extract_keywords(reference)
    return len(pred_kw & ref_kw) / len(pred_kw) if pred_kw else 0.0

def keyword_overlap_count(predicted, reference):
    """
    Total number of overlapping keywords.
    """
    return len(extract_keywords(predicted) & extract_keywords(reference))


# ---------- Semantic Similarity Function ----------
def factual_consistency_score(answer, source):
    """
    Cosine similarity between sentence embeddings of the answer and the source.
    """
    a_vec = embedding_pipeline(answer)[0]
    s_vec = embedding_pipeline(source)[0]
    a_mean = [sum(col) / len(col) for col in zip(*a_vec)]
    s_mean = [sum(col) / len(col) for col in zip(*s_vec)]
    return cosine_similarity([a_mean], [s_mean])[0][0]

OSError: [E050] Can't find model 'en_core_web_sm'. It doesn't seem to be a Python package or a valid path to a data directory.

In [12]:
!pip install spacy sklearn transformers

  Installing build dependencies ... done
  Getting requirements to build wheel ... error
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> [15 lines of output]
      The 'sklearn' PyPI package is deprecated, use 'scikit-learn'
      rather than 'sklearn' for pip commands.
      
      Here is how to fix this error in the main use cases:
      - use 'pip install scikit-learn' rather than 'pip install sklearn'
      - replace 'sklearn' by 'scikit-learn' in your pip requirements files
        (requirements.txt, setup.py, setup.cfg, Pipfile, etc ...)
      - if the 'sklearn' package is used by one of your dependencies,
        it would be great if you take some time to track which package uses
        'sklearn' instead of 'scikit-learn' and report it to their issue tracker
      - as a last resort, set the environment variable
        SKLEARN_ALLOW_DEPRECATED_SKLEARN_PACKAGE_INSTALL=True to avoid this error
   

In [4]:
import re

def num_claims(text):
    """
    Counts phrases that commonly introduce a claim.
    """
    claim_keywords = [
        r"\b(i believe|i think|it is clear|we must|it is evident|in my opinion)\b"
    ]
    return sum(len(re.findall(pat, text.lower())) for pat in claim_keywords)

def num_evidence(text):
    """
    Counts phrases that commonly introduce supporting evidence.
    """
    evidence_keywords = [
        r"\b(because|for example|for instance|this shows|the data suggest)\b"
    ]
    return sum(len(re.findall(pat, text.lower())) for pat in evidence_keywords)

def has_claim(text):
    """
    Returns True if the text includes at least one claim.
    """
    return num_claims(text) > 0

def has_evidence(text):
    """
    Returns True if the text includes at least one piece of evidence.
    """
    return num_evidence(text) > 0

def has_intro(text):
    """
    Returns True if the text begins with an introduction-like phrase.
    """
    intro_patterns = [
        r"^in this (essay|paper|article)",
        r"^the purpose of this",
        r"^this (essay|article|paper) will"
    ]
    return any(re.search(pat, text.lower()) for pat in intro_patterns)

def has_conclusion(text):
    """
    Returns True if the text ends with a concluding phrase.
    """
    conclusion_patterns = [
        r"\bin conclusion\b",
        r"\bto sum up\b",
        r"\bto conclude\b",
        r"\bin summary\b"
    ]
    return any(re.search(pat, text.lower()) for pat in conclusion_patterns)

def structure_score(text):
    """
    Returns a score between 0 and 1 based on presence of:
    introduction, claim, evidence, and conclusion.
    """
    parts = [has_intro(text), has_claim(text), has_evidence(text), has_conclusion(text)]
    return sum(parts) / 4

def persuasiveness_score(text):
    """
    Returns a weighted score combining claim and evidence counts.
    """
    claims = num_claims(text)
    evidence = num_evidence(text)
    return 0.6 * claims + 0.4 * evidence

In [11]:
from sentence_transformers import SentenceTransformer
from transformers import pipeline
from sklearn.metrics.pairwise import cosine_similarity
import nltk
nltk.download('punkt')
from nltk.tokenize import sent_tokenize
import numpy as np

# Load models
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
sentiment_model = pipeline("sentiment-analysis")

def coherence_score(dialogue):
    """
    Returns average cosine similarity between adjacent utterances.
    Input: dialogue as a list of strings (utterances).
    """
    if not isinstance(dialogue, list) or len(dialogue) < 2:
        return 0.0
    embeddings = embedding_model.encode(dialogue)
    similarities = [
        cosine_similarity([embeddings[i]], [embeddings[i+1]])[0][0]
        for i in range(len(embeddings) - 1)
    ]
    return float(np.mean(similarities))

def sentiment_flow(dialogue):
    """
    Returns the average sentiment change between utterances.
    Lower is smoother. Input is a list of utterances.
    """
    if not isinstance(dialogue, list) or len(dialogue) < 2:
        return 0.0
    sentiments = []
    for line in dialogue:
        score = sentiment_model(line)[0]['score']
        if sentiment_model(line)[0]['label'] == 'NEGATIVE':
            score = -score
        sentiments.append(score)
    diffs = [abs(sentiments[i] - sentiments[i+1]) for i in range(len(sentiments)-1)]
    return float(np.mean(diffs))

def relevance_to_prompt(response, prompt):
    """
    Returns cosine similarity between response and prompt.
    """
    embeddings = embedding_model.encode([prompt, response])
    return float(cosine_similarity([embeddings[0]], [embeddings[1]])[0][0])

[nltk_data] Downloading package punkt to /home/codespace/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f (https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.
Device set to use cpu


In [10]:
!pip install sentence-transformers transformers nltk scikit-learn


[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python3 -m pip install --upgrade pip


In [6]:
import nltk
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')

[nltk_data] Downloading package punkt to /home/codespace/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /home/codespace/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.


True

In [9]:
import nltk
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.probability import FreqDist
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import pronouncing
import numpy as np
import textstat

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

def narrative_coherence(text):
    """
    Computes average cosine similarity between adjacent sentences.
    """
    sentences = sent_tokenize(text)
    if len(sentences) < 2:
        return 0.0
    embeddings = embedding_model.encode(sentences)
    sims = [
        cosine_similarity([embeddings[i]], [embeddings[i+1]])[0][0]
        for i in range(len(embeddings) - 1)
    ]
    return float(np.mean(sims))

def emotional_arc(text):
    """
    Measures variance in sentiment across sentences (higher = more emotional shifts).
    """
    sentiment = nltk.sentiment.SentimentIntensityAnalyzer()
    sentences = sent_tokenize(text)
    scores = [sentiment.polarity_scores(s)['compound'] for s in sentences]
    return float(np.std(scores)) if scores else 0.0

def lexical_richness(text):
    """
    Measures type-token ratio (unique words / total words).
    """
    words = word_tokenize(text.lower())
    words = [w for w in words if w.isalpha()]
    return len(set(words)) / len(words) if words else 0.0

def rhyme_density(poem):
    """
    Measures fraction of end words in rhyming pairs.
    """
    lines = [line.strip() for line in poem.split("\n") if line.strip()]
    end_words = [line.split()[-1].lower() for line in lines if line.split()]
    rhymes = 0
    total = 0
    for i in range(len(end_words)):
        for j in range(i + 1, len(end_words)):
            total += 1
            if end_words[j] in pronouncing.rhymes(end_words[i]):
                rhymes += 1
    return rhymes / total if total else 0.0

def syllable_consistency(poem):
    """
    Measures standard deviation of syllables per line (lower = more consistent).
    """
    lines = [line.strip() for line in poem.split("\n") if line.strip()]
    syllables = [textstat.syllable_count(line) for line in lines]
    return float(np.std(syllables)) if syllables else 0.0

def thematic_similarity(poem, theme):
    """
    Measures cosine similarity between poem and theme embeddings.
    """
    embeddings = embedding_model.encode([poem, theme])
    return float(cosine_similarity([embeddings[0]], [embeddings[1]])[0][0])

/usr/local/python/3.12.1/lib/python3.12/site-packages/pronouncing/__init__.py:3: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream


In [8]:
!pip install nltk textstat pronouncing sentence-transformers scikit-learn


[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python3 -m pip install --upgrade pip
